# Processus de Poisson à intensité double-pic modulée par un facteur latent

On considère

$$
\lambda(t)=\lambda_0(t)X_t,
$$

où $\lambda_0(t)$ est la somme de deux cloches gaussiennes centrées à 9 h et 16 h, avec le temps exprimé en secondes depuis minuit.

Deux facteurs latents sont proposés :

1. un processus d'Ornstein–Uhlenbeck tronqué pour garantir la positivité ;
2. un processus CIR, positif par construction sous des conditions adaptées.

Les événements sont ensuite simulés par thinning.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import ArrayLike, NDArray

SECONDS_PER_HOUR = 3600.0
SECONDS_PER_MINUTE = 60.0

## Profil déterministe

$$
\lambda_0(t)
=
b
+
A_1\exp\left[-\frac{(t-9\times3600)^2}{2s_1^2}\right]
+
A_2\exp\left[-\frac{(t-16\times3600)^2}{2s_2^2}\right].
$$

In [ ]:
def double_gaussian_baseline(
    t: ArrayLike,
    base: float = 0.02,
    amplitude_9h: float = 0.25,
    amplitude_16h: float = 0.30,
    sigma_9h: float = 45.0 * SECONDS_PER_MINUTE,
    sigma_16h: float = 60.0 * SECONDS_PER_MINUTE,
) -> NDArray[np.float64]:
    if base < 0 or amplitude_9h < 0 or amplitude_16h < 0:
        raise ValueError("La base et les amplitudes doivent être positives.")
    if sigma_9h <= 0 or sigma_16h <= 0:
        raise ValueError("Les écarts-types doivent être strictement positifs.")

    t = np.asarray(t, dtype=float)
    m1 = 9.0 * SECONDS_PER_HOUR
    m2 = 16.0 * SECONDS_PER_HOUR

    return (
        base
        + amplitude_9h * np.exp(-0.5 * ((t - m1) / sigma_9h) ** 2)
        + amplitude_16h * np.exp(-0.5 * ((t - m2) / sigma_16h) ** 2)
    )

## Processus d'Ornstein–Uhlenbeck

$$
dX_t=\kappa(\theta-X_t)\,dt+\sigma\,dW_t.
$$

Comme $X_t$ peut devenir négatif, on utilise

$$
\lambda(t)=\lambda_0(t)\max(X_t,\varepsilon).
$$

In [ ]:
def simulate_ou_exact(
    time_grid: ArrayLike,
    kappa: float,
    theta: float,
    sigma: float,
    x0: float,
    seed: int | None = None,
) -> NDArray[np.float64]:
    times = np.asarray(time_grid, dtype=float)

    if np.any(np.diff(times) <= 0):
        raise ValueError("Les temps doivent être strictement croissants.")
    if kappa <= 0 or sigma < 0:
        raise ValueError("Paramètres OU invalides.")

    rng = np.random.default_rng(seed)
    x = np.empty(len(times), dtype=float)
    x[0] = x0

    for k in range(1, len(times)):
        dt = times[k] - times[k - 1]
        phi = np.exp(-kappa * dt)
        sd = sigma * np.sqrt((1.0 - np.exp(-2.0 * kappa * dt)) / (2.0 * kappa))
        x[k] = theta + phi * (x[k - 1] - theta) + sd * rng.normal()

    return x

In [ ]:
def simulate_thinning_from_grid(
    time_grid: ArrayLike,
    intensity_grid: ArrayLike,
    seed: int | None = None,
) -> NDArray[np.float64]:
    times = np.asarray(time_grid, dtype=float)
    intensity = np.asarray(intensity_grid, dtype=float)

    if len(times) != len(intensity):
        raise ValueError("Les deux tableaux doivent avoir la même longueur.")
    if np.any(np.diff(times) <= 0):
        raise ValueError("La grille doit être strictement croissante.")
    if np.any(intensity < 0) or np.any(~np.isfinite(intensity)):
        raise ValueError("L'intensité doit être positive et finie.")

    upper = float(np.max(intensity))
    if upper == 0:
        return np.empty(0, dtype=float)

    rng = np.random.default_rng(seed)
    t = times[0]
    end = times[-1]
    events = []

    while True:
        t += rng.exponential(1.0 / upper)
        if t >= end:
            break

        idx = np.searchsorted(times, t, side="right") - 1
        idx = min(idx, len(intensity) - 1)

        if rng.uniform() <= intensity[idx] / upper:
            events.append(t)

    return np.asarray(events, dtype=float)

In [ ]:
start = 8.0 * SECONDS_PER_HOUR
end = 18.0 * SECONDS_PER_HOUR
dt = 10.0

time_grid = np.arange(start, end + dt, dt)

baseline = double_gaussian_baseline(
    time_grid,
    base=0.02,
    amplitude_9h=0.25,
    amplitude_16h=0.30,
    sigma_9h=45.0 * SECONDS_PER_MINUTE,
    sigma_16h=60.0 * SECONDS_PER_MINUTE,
)

ou_path = simulate_ou_exact(
    time_grid,
    kappa=1.0 / (30.0 * SECONDS_PER_MINUTE),
    theta=1.0,
    sigma=0.015,
    x0=1.0,
    seed=123,
)

epsilon = 1e-4
intensity_ou = baseline * np.maximum(ou_path, epsilon)

events_ou = simulate_thinning_from_grid(
    time_grid,
    intensity_ou,
    seed=456,
)

print("Nombre d'événements OU :", len(events_ou))
print("Minimum du OU :", ou_path.min())

In [ ]:
hours = time_grid / SECONDS_PER_HOUR

plt.figure(figsize=(11, 5))
plt.plot(hours, baseline, label=r"$\lambda_0(t)$")
plt.plot(hours, intensity_ou, label=r"$\lambda(t)$ avec OU tronqué")
plt.xlabel("Heure")
plt.ylabel("Événements par seconde")
plt.title("Intensité double-pic modulée par un OU latent")
plt.legend()
plt.tight_layout()
plt.show()

## Alternative positive : processus CIR

$$
dX_t=\kappa(\theta-X_t)\,dt+\sigma\sqrt{X_t}\,dW_t.
$$

On utilise directement

$$
\lambda(t)=\lambda_0(t)X_t.
$$

La condition de Feller est

$$
2\kappa\theta\geq\sigma^2.
$$

La simulation ci-dessous utilise Euler avec troncature complète.

In [ ]:
def simulate_cir_full_truncation(
    time_grid: ArrayLike,
    kappa: float,
    theta: float,
    sigma: float,
    x0: float,
    seed: int | None = None,
) -> NDArray[np.float64]:
    times = np.asarray(time_grid, dtype=float)

    if np.any(np.diff(times) <= 0):
        raise ValueError("Les temps doivent être strictement croissants.")
    if kappa <= 0 or theta <= 0 or sigma < 0 or x0 < 0:
        raise ValueError("Paramètres CIR invalides.")

    rng = np.random.default_rng(seed)
    x = np.empty(len(times), dtype=float)
    x[0] = x0

    for k in range(1, len(times)):
        dt = times[k] - times[k - 1]
        x_pos = max(x[k - 1], 0.0)
        x[k] = (
            x[k - 1]
            + kappa * (theta - x_pos) * dt
            + sigma * np.sqrt(x_pos * dt) * rng.normal()
        )
        x[k] = max(x[k], 0.0)

    return x

In [ ]:
cir_path = simulate_cir_full_truncation(
    time_grid,
    kappa=1.0 / (30.0 * SECONDS_PER_MINUTE),
    theta=1.0,
    sigma=0.01,
    x0=1.0,
    seed=789,
)

intensity_cir = baseline * cir_path
events_cir = simulate_thinning_from_grid(
    time_grid,
    intensity_cir,
    seed=987,
)

print("Nombre d'événements CIR :", len(events_cir))
print("Minimum du CIR :", cir_path.min())

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(hours, baseline, label=r"$\lambda_0(t)$")
plt.plot(hours, intensity_cir, label=r"$\lambda(t)$ avec CIR")
plt.xlabel("Heure")
plt.ylabel("Événements par seconde")
plt.title("Intensité double-pic modulée par un facteur CIR")
plt.legend()
plt.tight_layout()
plt.show()

## Comptages sur la grille

Pour un futur filtre, on peut construire

$$
Y_k=N(t_{k+1})-N(t_k).
$$

In [ ]:
counts_ou, _ = np.histogram(events_ou, bins=time_grid)
counts_cir, _ = np.histogram(events_cir, bins=time_grid)

print("Premiers comptages OU :", counts_ou[:20])
print("Premiers comptages CIR :", counts_cir[:20])